In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

모델: gpt-5.6-luna


In [3]:
from neo4j import GraphDatabase

# os.getenv 의 두 번째 인자가 기본값이다. .env 에 값이 없으면 이 값으로 접속한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 연결이 안 되면 여기서 바로 에러가 난다. 뒤 셀까지 가지 않는다


def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # 값은 query 에 붙이지 않고 params 로 따로 넘긴다. 따옴표가 든 근거 문장도 안전하다
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

Neo4j 연결: bolt://localhost:7687


In [2]:
import json, os, re
from pathlib import Path
DATA_PATH=next((p for p in [Path.cwd()/'전처리/recipes_10000_preprocessed.jsonl',Path.cwd()/'recipes_10000_preprocessed.jsonl',Path.cwd().parent/'전처리/recipes_10000_preprocessed.jsonl'] if p.exists()),None)
assert DATA_PATH, 'recipes_10000_preprocessed.jsonl을 찾지 못했습니다.'
recipes=[]
with DATA_PATH.open(encoding='utf-8') as f:
    for seq,line in enumerate(f):
        if seq>=3000: break
        if line.strip():
            x=json.loads(line); x['_sequence']=seq+1; recipes.append(x)
print(DATA_PATH,len(recipes))

c:\Users\Playdata\Desktop\mle-01-p2-team2\전처리\recipes_10000_preprocessed.jsonl 3000
